*전체 11단계 중 2단계(데이터 수집)의 노트북입니다 — 기상청 ASOS 시간자료. 진행 기록은 리포의 `README.md`.*

# 02. 기상청 ASOS 시간자료 수집 — 한 번 실행 (서울 108 지점, 2023-01 ~ 2026-08)

**무엇을 하나** — 공공데이터포털 「기상청_지상(종관, ASOS) 시간자료 조회서비스」로 한 달치씩 받아 `data/raw/asos/asos_108_YYYYMM.csv`로 저장한다. 기온(ta)·습도(hm)·풍속(ws)·강수(rn)·일사(icsr)·일조(ss) 등이 들어 있다.
**횟수** — 44개월 = 44번 호출. 이 API는 하루 10,000건까지라 한 번에 끝난다. 이미 받은 달은 건너뛴다.
**지점** — 서울(108)로 시작한다. 묶음 2에서 지점을 더 넣을지 정한다.
**주의** — 인증키는 `.env`에서 읽는다(발전량 API와 같은 키). 실행 전 ⌘S로 저장.


In [1]:
import os                                    # 폴더·파일 경로를 다루는 기본 모듈
import time                                  # 호출 사이에 잠깐 쉬려고 쓴다
import requests                              # 인터넷으로 API를 부르는 라이브러리
import pandas as pd                          # 표(데이터프레임)를 다루는 라이브러리
from dotenv import load_dotenv, find_dotenv  # .env 파일에서 키를 읽는 도구

load_dotenv(find_dotenv())                   # 프로젝트 안의 .env 파일을 찾아 읽는다
KEY = os.environ["DATA_GO_KR_KEY"]           # 인증키 (발전량 API와 같은 키)

START = "2023-01"                            # 받을 첫 달
END   = "2026-08"                            # 받을 마지막 달
STN   = "108"                                # 지점 번호 — 108 = 서울

URL  = "https://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"  # ASOS 시간자료 API 주소
ROOT = os.path.dirname(find_dotenv())        # .env가 있는 폴더 = 프로젝트 루트
OUT  = os.path.join(ROOT, "data", "raw", "asos")  # 월별 파일을 모을 폴더
os.makedirs(OUT, exist_ok=True)              # 폴더가 없으면 만든다

months = pd.period_range(START, END, freq="M")   # 2023-01 ~ 2026-08 의 달 목록

def fetch_month(p):                          # 한 달치를 받아 표로 돌려주는 함수 (p = 달)
    s = p.start_time.strftime("%Y%m%d")      # 그 달의 첫날 (20230101 꼴)
    e = p.end_time.strftime("%Y%m%d")        # 그 달의 마지막 날
    params = {"serviceKey": KEY,             # 인증키
              "pageNo": 1,                   # 첫 페이지
              "numOfRows": 999,              # 한 번에 받을 최대 행 수 (한 달은 최대 744행)
              "dataType": "JSON",            # JSON으로 달라고 요청
              "dataCd": "ASOS",              # 자료 종류: 지상 종관 관측
              "dateCd": "HR",                # 시간 단위 자료
              "startDt": s, "startHh": "00", # 시작일·시작시각
              "endDt": e, "endHh": "23",     # 종료일·종료시각
              "stnIds": STN}                 # 지점 번호
    r = requests.get(URL, params=params, timeout=60)   # API를 부른다 (60초 넘으면 포기)
    r.raise_for_status()                     # 응답 코드가 200이 아니면 오류
    js = r.json()["response"]                # 응답 본체
    if js["header"]["resultCode"] != "00":   # "00"이 정상, 그 외는 오류 메시지가 온다
        raise RuntimeError(js["header"]["resultMsg"])   # 메시지를 오류로 올린다
    items = js["body"].get("items") or {}    # 데이터가 없으면 빈 딕셔너리
    rows = items.get("item") or []           # 행 목록
    return pd.DataFrame(rows)                # 표로 바꿔 돌려준다

for p in months:                             # 달마다 반복
    path = os.path.join(OUT, f"asos_{STN}_{p.strftime('%Y%m')}.csv")   # 저장할 파일 이름
    if os.path.exists(path):                 # 이미 받은 달이면
        continue                             # 건너뛴다
    try:
        df = fetch_month(p)                  # 한 달치 받기
    except Exception as e:                   # 실패하면
        print(f"{p} 실패: {e}")              # 어느 달이 왜 실패했는지 찍고
        continue                             # 다음 달로 넘어간다 (다시 실행하면 그 달만 다시 받는다)
    df.to_csv(path, index=False, encoding="utf-8-sig")   # 저장
    print(f"{p}: {len(df)}행")               # 달 × 행 수 (31일이면 744행이 정상)
    time.sleep(0.3)                          # 0.3초 쉰다

print(f"완료 · 폴더 안 파일 {len(os.listdir(OUT))}개 (목표 {len(months)}개)")


2023-01: 744행
2023-02: 672행
2023-03: 744행
2023-04: 720행
2023-05: 744행
2023-06: 720행
2023-07: 744행
2023-08: 744행
2023-09: 720행
2023-10: 744행
2023-11: 720행
2023-12: 744행
2024-01: 744행
2024-02: 696행
2024-03: 744행
2024-04: 720행
2024-05: 744행
2024-06: 720행
2024-07: 744행
2024-08: 744행
2024-09: 720행
2024-10: 744행
2024-11: 720행
2024-12: 744행
2025-01: 744행
2025-02: 672행
2025-03: 744행
2025-04: 720행
2025-05: 744행
2025-06: 720행
2025-07: 744행
2025-08: 744행
2025-09: 720행
2025-10: 744행
2025-11: 720행
2025-12: 744행
2026-01: 744행
2026-02: 672행
2026-03: 744행
2026-04: 720행
2026-05: 744행
2026-06: 720행
2026-07: 744행
2026-08: 744행
완료 · 폴더 안 파일 44개 (목표 44개)
